In [2]:
!pip install Sastrawi
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

import requests
import pandas as pd
import re
import time
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from google.colab import files

API_KEY = "27e9b5c8912261520e865e78774cb6e9"

keywords = ["kebakaran", "karhutla", "hutan", "lahan", "iklim", "cuaca", "emisi", "bencana"]
semua_publikasi = []
print("Mulai mengambil data dari BPS...")
for keyword in keywords:
    for page in range(1, 6):
        url = f"https://webapi.bps.go.id/v1/api/list/model/publication/lang/ind/domain/0000/keyword/{keyword}/page/{page}/key/{API_KEY}/"
        response = requests.get(url)

        if response.status_code == 200:
            data = response.json()
            if data['data-availability'] == 'available':
                for item in data['data'][1]:
                    semua_publikasi.append({
                        'judul': item['title'],
                        'tanggal_rilis': item['rl_date'],
                        'abstrak': item['abstract'],
                        'kata_kunci': keyword
                    })
        time.sleep(1)

df_teks = pd.DataFrame(semua_publikasi)
df_teks = df_teks.dropna(subset=['abstrak']).drop_duplicates(subset=['judul'])
print(f"Berhasil mengumpulkan {len(df_teks)} dokumen teks!")

print("Memulai Pra-pemrosesan NLP (Ini butuh waktu 1-3 menit)...")

factory = StemmerFactory()
stemmer = factory.create_stemmer()
stop_words = set(stopwords.words('indonesian'))

tambahan_stopword = ['bps', 'badan', 'pusat', 'statistik', 'publikasi', 'buku', 'data', 'tabel', 'halaman']
stop_words.update(tambahan_stopword)

def pra_pemrosesan(teks):
    teks = str(teks).lower()
    teks = re.sub(r'[^a-z\s]', '', teks)

    tokens = word_tokenize(teks)

    hasil = []
    for kata in tokens:
        if kata not in stop_words:
            kata_dasar = stemmer.stem(kata)
            if kata_dasar:
                hasil.append(kata_dasar)

    return " ".join(hasil)

df_teks['teks_bersih'] = df_teks['abstrak'].apply(pra_pemrosesan)

import re

def pembersihan_ekstra(teks):
    teks = re.sub(r'&[a-zA-Z]+;', ' ', str(teks))

    teks = teks.lower()
    teks = re.sub(r'[^a-z\s]', ' ', teks)

    tokens = word_tokenize(teks)

    hasil = []
    for kata in tokens:
        if kata not in stop_words and len(kata) > 2:
            kata_dasar = stemmer.stem(kata)
            if kata_dasar:
                hasil.append(kata_dasar)

    return " ".join(hasil)

print("Membersihkan ulang data teks...")
df_teks['teks_bersih'] = df_teks['abstrak'].apply(pembersihan_ekstra)

nama_file = 'Data_Teks_BPS_Karhutla_Bersih.csv'
df_teks.to_csv(nama_file, index=False)
files.download(nama_file)
print("Selesai! File yang lebih bersih sudah diunduh.")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Mulai mengambil data dari BPS...
Terkumpul 122 dokumen sebelum disaring relevansi
Berhasil mengumpulkan 110 dokumen teks yang relevan!
Memulai Pra-pemrosesan NLP (Ini butuh waktu 1-3 menit)...
Membersihkan ulang data teks...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Selesai! File yang lebih bersih sudah diunduh.
